# Backtracking Interpretability Pipeline

This notebook runs the full pipeline for probing reasoning model hidden states to predict answer correctness.

**API Options:**
- **OpenRouter API**: Generate CoT via API (no GPU needed for generation, only for hidden state extraction)
- **Local**: Run everything locally with HuggingFace transformers (requires GPU)

**Pipeline stages:**
1. Setup & data download
2. Generate Chain-of-Thought reasoning (API or local)
3. Chunk reasoning into segments
4. Label chunks with correctness/stability
5. Extract hidden states (requires local model)
6. Train probes

Based on [Zhang et al. (2025) "Verifier Probing"](https://arxiv.org/abs/2504.05419)

## 0. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone the repo (or upload your local copy)
!git clone https://github.com/YOUR_USERNAME/model-backtracking.git
%cd model-backtracking

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate datasets huggingface_hub sentencepiece
!pip install -q spacy scikit-learn tqdm pyyaml requests
!python -m spacy download en_core_web_sm

In [ ]:
# Optional: Login to HuggingFace for gated models (needed for local inference)
from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")  # Uncomment and add your token

# Verify OpenRouter API key is set (if using API backend)
import os
if os.environ.get("OPENROUTER_API_KEY"):
    print("✓ OPENROUTER_API_KEY is set")
else:
    print("⚠ OPENROUTER_API_KEY not set - set it in the next cell if using OpenRouter")

## 1. Configuration

Choose your API backend and dataset:

In [ ]:
# ============================================
# API BACKEND: Choose one
# ============================================
# Option 1: OpenRouter API (no GPU needed for generation)
API_BACKEND = "openrouter"
MODEL_NAME = "deepseek/deepseek-r1"  # OpenRouter model ID
# MODEL_NAME = "qwen/qwq-32b"  # Alternative

# Option 2: Local HuggingFace (requires GPU)
# API_BACKEND = "local"
# MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# For hidden state extraction, we need a local model
# This can be a smaller distilled version of the API model
LOCAL_MODEL_FOR_HIDDEN = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# ============================================
# DATASET: Choose one
# ============================================
# Option A: GSM8K (numeric math problems)
DATASET = "gsm8k_test"
ANSWER_TYPE = "numeric"

# Option B: MMLU-STEM (multiple choice)
# DATASET = "mmlu_stem"
# ANSWER_TYPE = "letter"

# Number of examples (set to None for full dataset)
MAX_EXAMPLES = 50

# ============================================
# API KEY (for OpenRouter)
# ============================================
import os
# Set your API key here or use environment variable
# os.environ["OPENROUTER_API_KEY"] = "your-key-here"

# Paths (auto-generated based on config)
model_slug = MODEL_NAME.replace("/", "_")
RAW_PATH = f"data/raw/{DATASET}.jsonl"
COT_PATH = f"data/cot/{model_slug}/{DATASET}_rollouts.jsonl"
CHUNKS_PATH = f"data/chunks/segmented_{DATASET}.jsonl"
LABELED_PATH = f"data/labeled/labeled_intermediate_{DATASET}.jsonl"
local_model_slug = LOCAL_MODEL_FOR_HIDDEN.replace("/", "_")
REPS_DIR = f"data/reps/{local_model_slug}/{DATASET}"

## 2. Download Dataset

In [ ]:
from datasets import load_dataset
from pathlib import Path
import json

Path("data/raw").mkdir(parents=True, exist_ok=True)

if DATASET == "gsm8k_test":
    # Download GSM8K test set
    ds = load_dataset("openai/gsm8k", "main", split="test")
    with open(RAW_PATH, "w") as f:
        for row in ds:
            f.write(json.dumps(row) + "\n")
    print(f"Saved {len(ds)} GSM8K examples to {RAW_PATH}")

elif DATASET == "mmlu_stem":
    # Download MMLU-STEM with choices formatting
    ds = load_dataset("TIGER-Lab/MMLU-STEM", split="test")
    INDEX_TO_LETTER = {0: "A", 1: "B", 2: "C", 3: "D"}
    
    with open(RAW_PATH, "w") as f:
        for row in ds:
            # Format question with choices
            question = row["question"].strip() + "\n\n"
            for i, choice in enumerate(row["choices"]):
                question += f"{INDEX_TO_LETTER[i]}) {choice}\n"
            
            processed = {
                "question": question.strip(),
                "answer": INDEX_TO_LETTER[row["answer"]],
                "answer_type": "letter",
                "subject": row["subject"],
            }
            f.write(json.dumps(processed) + "\n")
    print(f"Saved {len(ds)} MMLU-STEM examples to {RAW_PATH}")

else:
    print(f"Unknown dataset: {DATASET}")

## 3. Generate Chain-of-Thought

Using either OpenRouter API (no GPU) or local model:

In [ ]:
# Generate reasoning traces
import subprocess

cmd = [
    "python", "-m", "src.pipeline.generate_cot",
    "--model", MODEL_NAME,
    "--input", RAW_PATH,
    "--dataset", DATASET,
    "--question-field", "question",
    "--answer-field", "answer",
    "--answer-type", ANSWER_TYPE,
    "--max-new-tokens", "8192",
    "--api", API_BACKEND,
]

if API_BACKEND == "local":
    cmd.append("--trust-remote-code")
else:
    # OpenRouter: add rate limiting
    cmd.extend(["--rate-limit-delay", "1.0"])

if MAX_EXAMPLES:
    cmd.extend(["--max-examples", str(MAX_EXAMPLES)])

print("Running:", " ".join(cmd))
!{" ".join(cmd)}

In [ ]:
# Preview a sample
import json
with open(COT_PATH) as f:
    sample = json.loads(f.readline())

print("Question:", sample["question"][:200], "...")
print("\nReasoning (first 500 chars):")
print(sample.get("reasoning", sample.get("cot", ""))[:500])

## 4. Chunk Reasoning Traces

In [ ]:
!python -m src.pipeline.chunk_cot \
    --input {COT_PATH} \
    --dataset {DATASET}

In [ ]:
# Preview chunks
with open(CHUNKS_PATH) as f:
    sample = json.loads(f.readline())

print(f"Example has {len(sample['chunks'])} chunks")
for i, chunk in enumerate(sample["chunks"][:3]):
    print(f"\nChunk {i}: {chunk['text'][:150]}...")

## 5. Label Intermediate Answers

In [ ]:
# Label with regex-based extraction (fast)
# Use --answer-type to match the dataset type
!python -m src.pipeline.label_intermediate \
    --input {CHUNKS_PATH} \
    --dataset {DATASET} \
    --answer-type {ANSWER_TYPE} \
    --labeling-mode regex

In [ ]:
# Preview labels
with open(LABELED_PATH) as f:
    sample = json.loads(f.readline())

print(f"Ground truth: {sample['answer']}")
print(f"Final answer: {sample.get('final_answer')}")
print("\nChunk labels:")
for chunk in sample["chunks"][:5]:
    print(f"  Chunk {chunk['chunk_idx']}: answer={chunk.get('intermediate_answer')}, "
          f"correct={chunk.get('is_correct')}, stable={chunk.get('is_stable')}, "
          f"backtrack={chunk.get('is_backtrack_start')}")

## 6. Extract Hidden States

**Note:** This step requires a local model to extract hidden states. If you used the API for generation, we use a smaller local model (e.g., DeepSeek-R1-Distill-Qwen-1.5B) to extract hidden states from the same prompts + reasoning traces.

In [ ]:
# Extract hidden states using local model
# This is the only step that requires GPU/local model
!python -m src.pipeline.dump_hidden \
    --input {LABELED_PATH} \
    --model {LOCAL_MODEL_FOR_HIDDEN} \
    --dataset {DATASET} \
    --batch-size 2 \
    --trust-remote-code

In [ ]:
# Check output shards
import torch
from pathlib import Path

shards = sorted(Path(REPS_DIR).glob("part-*.pt"))
print(f"Found {len(shards)} shards in {REPS_DIR}")

if shards:
    data = torch.load(shards[0], weights_only=False)
    print(f"Hidden states shape: {data['hidden_states'].shape}")
    print(f"Metadata keys: {data['meta'][0].keys()}")
else:
    print("No shards found! Check the dump_hidden output above for errors.")

## 7. Train Probes

### 7a. Correctness Probe (Baseline)

In [ ]:
# Train correctness probe
!python -m src.probes.train \
    --data-dir {REPS_DIR} \
    --label-key is_correct \
    --hidden-size 0 \
    --epochs 100 \
    --lr 1e-4 \
    --output results/correctness_probe.pt \
    --results-json results/correctness_results.json

In [ ]:
# View results
with open("results/correctness_results.json") as f:
    results = json.load(f)

print("Correctness Probe Results:")
print(f"  ROC-AUC: {results['final_metrics']['roc_auc']:.4f}")
print(f"  Accuracy: {results['final_metrics']['accuracy']:.4f}")
print(f"  F1: {results['final_metrics']['f1']:.4f}")

### 7b. Lookahead Probe (Predict Next Chunk's Correctness)

In [ ]:
# Train lookahead probe - predicts if NEXT answer will be correct
!python -m src.probes.delta_probe \
    --data-dir {REPS_DIR} \
    --mode lookahead \
    --hidden-size 0 \
    --epochs 100 \
    --lr 1e-4 \
    --output results/lookahead_probe.pt \
    --results-json results/lookahead_results.json

In [ ]:
with open("results/lookahead_results.json") as f:
    results = json.load(f)

print("Lookahead Probe Results (predict next chunk correctness):")
print(f"  ROC-AUC: {results['final_metrics']['roc_auc']:.4f}")
print(f"  Accuracy: {results['final_metrics']['accuracy']:.4f}")

### 7c. Stability Probe (Predict if Answer Will Change)

In [ ]:
# Train stability probe
!python -m src.probes.delta_probe \
    --data-dir {REPS_DIR} \
    --mode stability \
    --hidden-size 0 \
    --epochs 100 \
    --lr 1e-4 \
    --output results/stability_probe.pt \
    --results-json results/stability_results.json

### 7d. Grid Search (Optional)

In [ ]:
# Full hyperparameter grid search
!python scripts/train_probes.py \
    --data-dir {REPS_DIR} \
    --label-key is_correct \
    --quick  # Use --quick for faster search, remove for full grid

## 8. Analysis

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Compare probe results
results_dir = Path("results")
probes = []

for name in ["correctness", "lookahead", "stability"]:
    path = results_dir / f"{name}_results.json"
    if path.exists():
        with open(path) as f:
            data = json.load(f)
            probes.append({
                "name": name,
                "roc_auc": data["final_metrics"].get("roc_auc", 0),
                "accuracy": data["final_metrics"].get("accuracy", 0),
                "f1": data["final_metrics"].get("f1", 0),
            })

if probes:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    x = range(len(probes))
    width = 0.25
    
    ax.bar([i - width for i in x], [p["roc_auc"] for p in probes], width, label="ROC-AUC")
    ax.bar([i for i in x], [p["accuracy"] for p in probes], width, label="Accuracy")
    ax.bar([i + width for i in x], [p["f1"] for p in probes], width, label="F1")
    
    ax.set_xticks(x)
    ax.set_xticklabels([p["name"] for p in probes])
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.set_title("Probe Performance Comparison")
    plt.tight_layout()
    plt.show()

## 9. Save Results to Drive (Optional)

In [ ]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Copy results
# !cp -r results /content/drive/MyDrive/backtracking_results/